In [28]:
from dotenv import load_dotenv
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.runnables import RunnableParallel , RunnableLambda , RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

In [26]:
load_dotenv()

embedding_model = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-2-preview')

model = ChatGoogleGenerativeAI(model= 'gemini-3.1-flash-lite-preview')

parser = StrOutputParser()

In [6]:
loader = WikipediaLoader(
    query = 'Agentic AI' , 
    lang = 'en' , 
    load_max_docs = 10 
)

In [7]:
doc = loader.load()

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500 , 
    chunk_overlap = 400
)

In [11]:
chunk = splitter.split_documents(doc)

In [12]:
len(chunk)

32

In [16]:
vector_store = FAISS.from_documents(
    embedding = embedding_model , 
    documents=chunk
)

In [18]:
retriever = vector_store.as_retriever()

In [19]:
def format(retriever_docs):
    context = '\n\n'.join(d.page_content for d in retriever_docs)

    return context

In [21]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format) , 
    'query' : RunnablePassthrough()
})

In [22]:
parallel_chain.get_graph().print_ascii()

             +------------------------------+          
             | Parallel<context,query>Input |          
             +------------------------------+          
                    **               ***               
                 ***                    **             
               **                         ***          
+----------------------+                     **        
| VectorStoreRetriever |                      *        
+----------------------+                      *        
            *                                 *        
            *                                 *        
            *                                 *        
       +--------+                     +-------------+  
       | format |                     | Passthrough |  
       +--------+***                  +-------------+  
                    **               **                
                      ***         ***                  
                         **     **              

In [27]:
prompt = PromptTemplate(
    template='Hey , Assistant i am sharing a context and a query just extract a appropriate response from the context is context is insufficient then just send the context is insufficient. \n context -> {context} \n query - > {query}'
)

In [29]:
chain_2 = prompt | model | parser

chain =RunnableSequence( parallel_chain , chain_2 )

In [32]:
query = 'What is a Agents'

In [33]:
chain.invoke(query)

'In artificial intelligence, an intelligent agent is an entity that perceives its environment, takes actions autonomously to achieve goals, and may improve its performance through machine learning or by acquiring knowledge. \n\nBeyond this general definition, agents (also referred to as agentic AI or compound AI systems) are characterized by:\n\n*   **Autonomy:** They act independently of user supervision and can operate in complex environments over extended periods.\n*   **Goal-Directed Behavior:** They operate based on objective functions (such as reward or fitness functions) and are designed to create and execute plans to maximize the expected value of these goals.\n*   **Complexity:** They can range from simple systems (like a thermostat) to highly complex entities (like a human, a firm, or a state).\n*   **Attributes:** AI agents often feature natural language interfaces, memory systems, and the ability to integrate with software tools or planning systems. Their control flow is fr